In [1]:
#LIBARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [3]:
#
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [4]:
#
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [5]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [6]:
x_train = train.drop(columns=["id","addicted_label"])
y_train = train["addicted_label"]

x_test = test.drop(columns=["id"])

In [7]:
categoric_cols = ['gender','stress_level','academic_work_impact']

for col in categoric_cols:
    x_train[col] = x_train[col].astype('category')
    x_test[col] = x_test[col].astype('category')

In [8]:
# validation
x_trn, x_val, y_trn, y_val = train_test_split(x_train, y_train, test_size=0.2, 
                                              stratify = y_train, random_state=42)

In [9]:
#light GBM
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate = 0.03,
    max_depth = 6,
    num_leaves=31,
    random_state = 42,
    n_jobs=-1
)

lgb_model.fit(x_trn, y_trn, eval_set=[(x_val, y_val)],
       eval_metric='auc',
       callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
       )

C:\Users\Admin\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.03
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [11]:
# 5. Evaluate validation metrics
val_probs = lgb_model.predict_proba(x_val)[:, 1]
print('Validation ROC AUC:', roc_auc_score(y_val, val_probs))
print('Validation Log Loss:', log_loss(y_val, val_probs))

Validation ROC AUC: 0.9594002883784498
Validation Log Loss: 0.23446364263502598


In [13]:
# 6. Predict probabilities on test set
test_predictions = lgb_model.predict_proba(x_test)[:, 1]

In [14]:
# 1. Instantiate LightGBM using the optimal number of trees from early stopping
final_lgb = lgb.LGBMClassifier(
    n_estimators=lgb_model.best_iteration_,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

# 2. Fit on 100% of x_train
final_lgb.fit(x_train, y_train)

# 3. Predict test probabilities
test_probability = final_lgb.predict_proba(x_test)[:, 1]

[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1955
[LightGBM] [Info] Number of data points in the train set: 691369, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892590
[LightGBM] [Info] Start training from score 0.892590


In [15]:
# 4. Create output DataFrame
output_df = pd.DataFrame({
    'id': test['id'],
    'predicted_probability': test_probability
})

In [16]:
output_df.head(10)

,id,predicted_probability
0,691369,0.998273
1,691370,0.958129
2,691371,0.964452
3,691372,0.982076
4,691373,0.995584
5,691374,0.818037
6,691375,0.915278
7,691376,0.377725
8,691377,0.978569
9,691378,0.441845


In [17]:
output_df.to_csv('submission.csv', index=False)